# Standard RNN (slow): Parameter grid search for each architecture.

In [ ]:
import os, itertools

from load_MNIST     import make_timeseries_loaders
from Utilities      import append_row, run_one
from RNN_functions  import StandardRNN, rnn_prepare_batch

# ---------------- Settings ----------------
EPOCHS      = 30
SEEDS       = [123, 456, 789]
NUM_WORKERS = 0
PRINT_STATS = False

OUT_DIR = "./out/gridsearch"
os.makedirs(OUT_DIR, exist_ok=True)

# Match RNN nodes
HIDDEN_SIZES = [11]

# Parameter grid
LRS = [3e-4, 1e-3, 3e-3]        # Learning rate
WDS = [0.0, 1e-3, 1e-2]         # Rate decay
LSS = [0.0, 0.05]               # Label smoothing

if __name__ == "__main__":
    loaders = {
        s: make_timeseries_loaders(
            batch_size=128,
            val_size=5000,
            seed=s,
            num_workers=NUM_WORKERS,
            print_stats=PRINT_STATS,
        )
        for s in SEEDS
    }

    cols = [
        "arch_name","hidden_size",
        "seed","lr","weight_decay","label_smoothing",
        "best_epoch","best_val_acc",
    ]

    for H in HIDDEN_SIZES:
        arch_name = f"RNN_h{H}"
        out_csv = os.path.join(OUT_DIR, f"rnn_gridsearch_h{H}.csv")
        print(f"\n===== {arch_name} =====")

        for lr, wd, ls in itertools.product(LRS, WDS, LSS):
            for seed in SEEDS:
                model = StandardRNN(n_classes=10, hidden_size=H)
                row = run_one(
                    loaders=loaders,
                    model_fn=lambda H=H: StandardRNN(n_classes=10, hidden_size=H),
                    prepare_batch=rnn_prepare_batch,
                    arch_name=arch_name,
                    seed=seed,
                    lr=lr,
                    wd=wd,
                    ls=ls,
                    epochs=EPOCHS,
                    extra_row={"hidden_size": H},
                )
                append_row(out_csv, cols, row)
                print(arch_name, "seed", seed, "lr", lr, "wd", wd, "ls", ls, "| val", row["best_val_acc"])
                print("Saved to:", out_csv)

# Standard RNN (slow): Run 10 instances of the Standard RNN architecture at the optimal hyperparameters.

In [ ]:
import os
import numpy as np

from load_MNIST      import make_timeseries_loaders
from Utilities       import set_seed, append_row, read_best_params, compute_confusion_matrix
from RNN_functions   import StandardRNN, rnn_prepare_batch
from train           import TrainConfig, Trainer

# ---------------- Settings ----------------
HIDDEN_SIZES = [11]

BEST_CSV   = "./out/best_runs/best_hparams_from_grid.csv"
OUT_DIR    = "./out/best_runs"
os.makedirs(OUT_DIR, exist_ok=True)

CM_DIR     = os.path.join(OUT_DIR, "confusion_matrices")
os.makedirs(CM_DIR, exist_ok=True)

OUT_CSV    = os.path.join(OUT_DIR, "rnn_best10runs.csv")

EPOCHS     = 30
EVAL_SEEDS = list(range(101, 111))   # 10 runs

# keep loaders/training consistent
BATCH_SIZE  = 128
VAL_SIZE    = 5000
NUM_WORKERS = 0
PRINT_STATS = False

if __name__ == "__main__":
    best = read_best_params(BEST_CSV)

    # Start fresh each time
    if os.path.exists(OUT_CSV):
        os.remove(OUT_CSV)

    cols = [
        "arch_name", "hidden_size",
        "seed",
        "lr", "weight_decay", "label_smoothing",
        "best_epoch", "best_val_acc",
        "test_loss", "test_acc",
    ]

    for H in HIDDEN_SIZES:
        arch_name = f"RNN_h{H}"
        hp = best[arch_name]
        print(f"\n===== {arch_name} (hidden_size={H}) =====")
        print("Using best hparams:", hp)

        for seed in EVAL_SEEDS:
            set_seed(seed)

            train_loader, val_loader, test_loader = make_timeseries_loaders(
                batch_size=BATCH_SIZE,
                val_size=VAL_SIZE,
                seed=seed,
                num_workers=NUM_WORKERS,
                print_stats=PRINT_STATS,
            )

            model = StandardRNN(n_classes=10, hidden_size=H)

            cfg = TrainConfig(
                epochs=EPOCHS,
                lr=hp["lr"],
                weight_decay=hp["weight_decay"],
                label_smoothing=hp["label_smoothing"],
                seed=seed,
                history_csv=False,
            )
            os.makedirs(cfg.ckpt_dir, exist_ok=True)
            os.makedirs(cfg.log_dir, exist_ok=True)

            trainer = Trainer(cfg, prepare_batch=rnn_prepare_batch)
            results = trainer.fit(
                model, train_loader, val_loader, test_loader,
                run_name=f"{arch_name}_BEST_seed{seed}"
            )

            # Compute + save confusion matrix on test set (post-fit; assumes trainer leaves model at best checkpoint for test)
            cm = compute_confusion_matrix(model, test_loader, rnn_prepare_batch, n_classes=10)
            cm_path = os.path.join(CM_DIR, f"confmat_{arch_name}_seed{seed}.npy")
            np.save(cm_path, cm)

            row = dict(
                arch_name=arch_name,
                hidden_size=H,
                seed=seed,
                lr=hp["lr"],
                weight_decay=hp["weight_decay"],
                label_smoothing=hp["label_smoothing"],
                best_epoch=results["best_epoch"],
                best_val_acc=results["best_val_acc"],
                test_loss=results["test"]["loss"],
                test_acc=results["test"]["acc"],
            )

            append_row(OUT_CSV, cols, row)
            print(arch_name, "seed", seed, "| test_acc", row["test_acc"])

    print("\nSaved results:", OUT_CSV)